In [11]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

adult = fetch_openml(name='adult', version=2, as_frame=True, parser='auto')
df = adult.frame.copy()

df.columns = [c.replace('-', '_') for c in df.columns]

df = df.rename(columns={'class': 'income'})

df.dropna(subset=['workclass', 'occupation', 'native_country'], inplace=True)

df['income'] = (
    df['income'].astype(str).str.strip().str.replace('.', '', regex=False) == '>50K'
).astype(int)

df.drop(columns=['fnlwgt'], inplace=True)

X = df.drop(columns=['income']).copy()
y = df['income']

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Egitim seti boyutu : {X_train.shape}')
print(f'Test seti boyutu   : {X_test.shape}')
print(f'\nTest sinif dagilimi:')
print(y_test.value_counts(normalize=True).rename({0: '<=50K', 1: '>50K'}).to_string())
print(f'\nKullanilan ozellikler ({len(X.columns)}):')
print(X.columns.tolist())

Egitim seti boyutu : (36177, 13)
Test seti boyutu   : (9045, 13)

Test sinif dagilimi:
income
<=50K    0.752128
>50K     0.247872

Kullanilan ozellikler (13):
['age', 'workclass', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country']


In [12]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

def compute_metrics(model, X_tr, y_tr, X_te, y_te):
    """Modeli egit, sure ol ve temel metrikleri donduret."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    return {
        'Accuracy' : accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred),
        'Recall'   : recall_score(y_te, y_pred),
        'F1 Score' : f1_score(y_te, y_pred),
        'ROC-AUC'  : roc_auc_score(y_te, y_prob),
        'Train Time (s)': round(elapsed, 2)
    }

# random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_metrics = compute_metrics(rf, X_train, y_train, X_test, y_test)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_metrics = compute_metrics(gb, X_train, y_train, X_test, y_test)

results_df = pd.DataFrame(
    {'Random Forest': rf_metrics, 'Gradient Boosting': gb_metrics}
).T

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
print('=== Metrik Karsilastirmasi ===')
print(results_df[metric_cols + ['Train Time (s)']].to_string())

=== Metrik Karsilastirmasi ===
                   Accuracy  Precision    Recall  F1 Score   ROC-AUC  Train Time (s)
Random Forest      0.844555   0.714139  0.621766  0.664759  0.892888            1.04
Gradient Boosting  0.862023   0.791667  0.601695  0.683730  0.917876            5.84


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

RF_COLOR = '#4C72B0'
GB_COLOR = '#DD8452'

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

# metrik karşılaştırması
rf_vals = [rf_metrics[m] for m in metric_cols]
gb_vals = [gb_metrics[m] for m in metric_cols]

x = np.arange(len(metric_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_rf = ax.bar(x - width/2, rf_vals, width, label='Random Forest',
                 color=RF_COLOR, edgecolor='white')
bars_gb = ax.bar(x + width/2, gb_vals, width, label='Gradient Boosting',
                 color=GB_COLOR, edgecolor='white')

for bar in bars_rf:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars_gb:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_ylim(0.50, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(metric_cols)
ax.set_ylabel('Score')
ax.set_title('Random Forest ve Gradient Boosting: Metrik Karşılaştırması')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('metrik_karsilastirma.png', dpi=150)
plt.close(fig)
print('Kaydedildi: metrik_karsilastirma.png')

#  belirleyici özellikler
feature_names = X.columns.tolist()
top_n = 8

def top_features(model, n):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1][:n]
    return [feature_names[i] for i in idx], imp[idx]

rf_feats, rf_imp = top_features(rf, top_n)
gb_feats, gb_imp = top_features(gb, top_n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(rf_feats[::-1], rf_imp[::-1], color=RF_COLOR, edgecolor='white')
ax1.set_title('Random Forest — Top 8 Features')
ax1.set_xlabel('Importance')
ax1.grid(axis='x', alpha=0.3)

ax2.barh(gb_feats[::-1], gb_imp[::-1], color=GB_COLOR, edgecolor='white')
ax2.set_title('Gradient Boosting — Top 8 Features')
ax2.set_xlabel('Importance')
ax2.grid(axis='x', alpha=0.3)

fig.suptitle('Random Forest vs Gradient Boosting: Belirleyici Özellikler',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('feature_importance.png', dpi=150)
plt.close(fig)
print('Kaydedildi: feature_importance.png')

# eğitim süresi karşılaştırma
times = [rf_metrics['Train Time (s)'], gb_metrics['Train Time (s)']]
labels = ['Random Forest', 'Gradient Boosting']
colors = [RF_COLOR, GB_COLOR]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, times, color=colors, edgecolor='white', width=0.4)

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{t:.2f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Eğitim Süresi (saniye)')
ax.set_title('Random Forest vs Gradient Boosting: Eğitim Süresi')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('egitim_suresi.png', dpi=150)
plt.close(fig)
print('Kaydedildi: egitim_suresi.png')

Kaydedildi: metrik_karsilastirma.png
Kaydedildi: feature_importance.png
Kaydedildi: egitim_suresi.png
